# 04 — Comparator Metric Extraction and Review Workbook

Purpose: mine targeted candidate metrics from recovered comparator full texts and produce a human-verification workbook.

This notebook does **not** create findings. It creates candidate metric rows with source quotes/page markers for RA/researcher verification.

In [ ]:
# Optional first-time setup:
# %pip install -q pandas openpyxl

from pathlib import Path
from datetime import datetime
import math
import re
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "Research_Workflow").is_dir() and (p.name == "DC_job_potential" or (p / "Media_Article").exists()):
            return p
    candidate = Path.cwd().resolve() / "DC_job_potential"
    if (candidate / "Research_Workflow").is_dir():
        return candidate
    raise RuntimeError("Could not locate DC_job_potential root. Run inside the project folder.")


ROOT = find_project_root(Path.cwd())
WORKFLOW = ROOT / "Research_Workflow"
EXTRACTION = WORKFLOW / "02_Source_Extraction"
DATA_DIR = EXTRACTION / "data"
REVIEW_DIR = DATA_DIR / "coding_outputs" / "comparator_review"
TEXT_DIR = DATA_DIR / "comparator_sources" / "fulltext"
OUT_DIR = REVIEW_DIR / "metric_extraction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

INVENTORY_PATH = REVIEW_DIR / "comparator_fulltext_inventory.csv"
REVIEW_TEMPLATE_PATH = REVIEW_DIR / "comparator_metric_review_template.csv"

print("ROOT:", ROOT)
print("INVENTORY:", INVENTORY_PATH)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# Controls
MIN_CONTEXT_SCORE = 2
MAX_CANDIDATES_PER_SOURCE_METRIC = 80
MAX_REVIEW_ROWS = 500

inventory = pd.read_csv(INVENTORY_PATH)
review_template = pd.read_csv(REVIEW_TEMPLATE_PATH)

usable = inventory[inventory.get("usable_for_metric_extraction", False).astype(bool)].copy()
meta = review_template.set_index("source_id").to_dict("index") if "source_id" in review_template.columns else {}

print("inventory rows:", len(inventory))
print("usable rows:", len(usable))
usable[["priority", "source_id", "title", "words", "text_file"]]

In [ ]:
NUM = r"(?P<num>\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)"
MONEY = r"(?P<money>(?:A\$|AUD|US\$|USD|\$|€|£)\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s?(?:billion|bn|million|m|trillion|tn)?)"

PATTERNS = [
    ("construction_jobs_peak_or_headcount", re.compile(NUM + r"\s*(?:peak\s*)?(?:construction\s*)?(?:workers?|jobs?|employees?|FTEs?)\b", re.I), ["construction", "build", "site", "worker", "job", "employment"]),
    ("operational_jobs_fte", re.compile(NUM + r"\s*(?:permanent|operational|operations|ongoing|direct)?\s*(?:jobs?|workers?|employees?|FTEs?|staff)\b", re.I), ["operation", "operational", "permanent", "ongoing", "staff", "employment"]),
    ("construction_job_years", re.compile(NUM + r"\s*(?:job[- ]?years?|person[- ]?years?|FTE[- ]?years?|worker[- ]?years?)\b", re.I), ["job", "employment", "construction", "person", "worker"]),
    ("jobs_per_capex", re.compile(NUM + r"\s*(?:jobs?|job[- ]?years?|FTEs?|employees?)\s*(?:per|/)\s*(?:\$|US\$|USD|A\$|AUD)?\s?\d?\s?(?:million|m|billion|bn)\b", re.I), ["per", "million", "billion", "investment", "job"]),
    ("capex_or_investment", re.compile(MONEY, re.I), ["investment", "capital", "capex", "cost", "expenditure", "project"]),
    ("capacity_mw", re.compile(NUM + r"\s*(?:MW|megawatts?)\b", re.I), ["mw", "megawatt", "capacity", "power", "energy"]),
    ("capacity_gwh", re.compile(NUM + r"\s*(?:GWh|gigawatt[- ]?hours?)\b", re.I), ["gwh", "gigawatt", "capacity", "battery"]),
    ("employment_per_capacity", re.compile(NUM + r"\s*(?:jobs?|FTEs?|employees?)\s*(?:per|/)\s*(?:MW|GWh|GW)\b", re.I), ["job", "employment", "capacity", "mw", "gwh"]),
    ("gfa_m2", re.compile(NUM + r"\s*(?:m2|m²|sqm|square metres|square meters)\b", re.I), ["floor", "area", "building", "gross", "gfa"]),
    ("gfa_sqft", re.compile(NUM + r"\s*(?:sq\.?\s*ft|square feet|sf)\b", re.I), ["floor", "area", "building", "gross", "gfa"]),
    ("duration_months", re.compile(NUM + r"\s*(?:months?|month)\b", re.I), ["construction", "duration", "build", "project", "period"]),
    ("duration_years", re.compile(NUM + r"\s*(?:years?|yr)\b", re.I), ["construction", "duration", "build", "project", "period", "employment"]),
    ("io_multiplier", re.compile(NUM + r"\s*(?:x|times|multiplier)\b", re.I), ["multiplier", "input-output", "input output", "indirect", "induced", "economic impact"]),
    ("percent_effect_or_share", re.compile(NUM + r"\s*%", re.I), ["employment", "job", "investment", "construction", "capital", "share", "effect", "impact", "increase"]),
]

SOURCE_HINTS = {
    "direct": ["direct"],
    "indirect_or_induced": ["indirect", "induced", "supported", "multiplier", "supply chain"],
    "construction": ["construction", "build", "site", "project delivery"],
    "operation": ["operation", "operational", "permanent", "ongoing", "staff"],
    "projected": ["forecast", "estimate", "projected", "model", "expected", "scenario"],
    "realised_or_empirical": ["observed", "realised", "realized", "ex post", "empirical", "actual", "audit"],
}

HIGH_VALUE_METRICS = {
    "construction_jobs_peak_or_headcount", "operational_jobs_fte", "construction_job_years",
    "jobs_per_capex", "employment_per_capacity", "capex_or_investment", "capacity_mw", "capacity_gwh",
    "io_multiplier",
}

def parse_num(value):
    if value is None:
        return None
    raw = str(value).replace(",", "")
    m = re.search(r"\d+(?:\.\d+)?", raw)
    if not m:
        return None
    n = float(m.group(0))
    low = raw.lower()
    if "trillion" in low or " tn" in low:
        n *= 1_000_000_000_000
    elif "billion" in low or " bn" in low:
        n *= 1_000_000_000
    elif "million" in low or re.search(r"\bm\b", low):
        n *= 1_000_000
    return n

def source_flags(text):
    low = text.lower()
    flags = []
    for name, terms in SOURCE_HINTS.items():
        if any(t in low for t in terms):
            flags.append(name)
    return "; ".join(flags)

def context_score(sentence, terms):
    low = sentence.lower()
    score = sum(1 for t in terms if t.lower() in low)
    if any(t in low for t in ["table", "figure", "appendix"]):
        score += 1
    if any(t in low for t in ["reference", "copyright", "downloaded", "journal homepage"]):
        score -= 2
    return score

def split_units(text):
    text = re.sub(r"\s+", " ", text)
    parts = re.split(r"(?<=[.!?])\s+|\n+|(?=\[\[PAGE \d+\]\])", text)
    return [p.strip() for p in parts if p.strip()]

def page_for(unit, current):
    m = re.search(r"\[\[PAGE\s+(\d+)\]\]", unit)
    return m.group(1) if m else current

print("Pattern library ready:", len(PATTERNS), "patterns")

In [ ]:
rows = []

for _, src in usable.iterrows():
    sid = src["source_id"]
    path = Path(src["text_file"])
    if not path.exists():
        continue
    text = path.read_text(encoding="utf-8", errors="ignore")
    units = split_units(text)
    page = ""
    per_source_counts = {}
    src_meta = meta.get(sid, {})
    for idx, unit in enumerate(units):
        page = page_for(unit, page)
        clean = re.sub(r"\[\[PAGE\s+\d+\]\]", "", unit).strip()
        if not re.search(r"\d", clean):
            continue
        if len(clean) < 30:
            continue
        prev_unit = units[idx-1] if idx > 0 else ""
        next_unit = units[idx+1] if idx + 1 < len(units) else ""
        context = re.sub(r"\s+", " ", " ".join([prev_unit, clean, next_unit])).strip()
        for metric, rx, terms in PATTERNS:
            key = (sid, metric)
            if per_source_counts.get(key, 0) >= MAX_CANDIDATES_PER_SOURCE_METRIC:
                continue
            score = context_score(context, terms)
            if score < MIN_CONTEXT_SCORE:
                continue
            for m in rx.finditer(clean):
                raw_value = m.groupdict().get("num") or m.groupdict().get("money") or m.group(0)
                rows.append({
                    "source_id": sid,
                    "asset_type": src_meta.get("asset_type", ""),
                    "title": src.get("title", src_meta.get("project_or_source_name", "")),
                    "source_url": src_meta.get("source_url", ""),
                    "metric": metric,
                    "raw_value": raw_value,
                    "numeric_value": parse_num(raw_value),
                    "page_or_location": f"page {page}" if page else "unknown",
                    "context_score": score,
                    "phase_and_job_type_flags": source_flags(context),
                    "context_quote": context[:1200],
                    "text_file": str(path),
                    "source_raw_path": src.get("source_raw_path", ""),
                    "verification_status": "candidate_unverified",
                    "priority_for_review": "high" if metric in HIGH_VALUE_METRICS and score >= 3 else "normal",
                })
                per_source_counts[key] = per_source_counts.get(key, 0) + 1

candidates = pd.DataFrame(rows)
if not candidates.empty:
    candidates = candidates.sort_values(["priority_for_review", "source_id", "metric", "context_score"], ascending=[True, True, True, False])

cand_path = OUT_DIR / "comparator_candidate_metric_extractions.csv"
candidates.to_csv(cand_path, index=False)
print("Wrote", cand_path)
print("candidate rows:", len(candidates))
if not candidates.empty:
    display(candidates.groupby(["source_id", "metric"]).size().sort_values(ascending=False).head(40).reset_index(name="rows"))

In [ ]:
# Build a compact review workbook. Keep all candidates in one sheet, and a prioritised slice in another.
if candidates.empty:
    priority_review = pd.DataFrame()
else:
    high = candidates[candidates["priority_for_review"].eq("high")].copy()
    priority_review = high.sort_values(["source_id", "metric", "context_score"], ascending=[True, True, False]).head(MAX_REVIEW_ROWS)
    priority_review = priority_review.copy()
    priority_review.insert(0, "review_id", [f"cmp_metric_{i+1:04d}" for i in range(len(priority_review))])
    priority_review["ra_decision"] = "not_started"
    priority_review["verified_value"] = ""
    priority_review["verified_metric_definition"] = ""
    priority_review["direct_vs_total"] = ""
    priority_review["projected_vs_realised"] = ""
    priority_review["notes"] = ""

source_summary = pd.DataFrame()
if not candidates.empty:
    source_summary = candidates.groupby(["source_id", "title", "asset_type"]).agg(
        candidate_rows=("metric", "size"),
        high_priority_rows=("priority_for_review", lambda s: int((s == "high").sum())),
        metric_types=("metric", lambda s: "; ".join(sorted(set(s)))),
    ).reset_index().sort_values(["high_priority_rows", "candidate_rows"], ascending=False)

readme = pd.DataFrame([
    {"field": "created_at", "value": datetime.now().isoformat(timespec="seconds")},
    {"field": "purpose", "value": "Comparator metric candidate extraction from recovered full texts."},
    {"field": "rule", "value": "Rows are candidate leads only. Verify exact source quote/page/table before using any metric."},
    {"field": "source_inventory", "value": str(INVENTORY_PATH)},
    {"field": "candidate_rows", "value": len(candidates)},
    {"field": "priority_review_rows", "value": len(priority_review)},
])

xlsx_path = OUT_DIR / "comparator_metric_review_workbook.xlsx"
priority_csv = OUT_DIR / "comparator_priority_metric_review.csv"
summary_csv = OUT_DIR / "comparator_metric_source_summary.csv"

priority_review.to_csv(priority_csv, index=False)
source_summary.to_csv(summary_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    readme.to_excel(writer, sheet_name="README", index=False)
    source_summary.to_excel(writer, sheet_name="source_summary", index=False)
    priority_review.to_excel(writer, sheet_name="priority_metric_review", index=False)
    candidates.to_excel(writer, sheet_name="all_candidate_metrics", index=False)
    usable.to_excel(writer, sheet_name="fulltext_inventory", index=False)

try:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation
    wb = load_workbook(xlsx_path)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = PatternFill("solid", fgColor="D9EAF7")
        for col in ws.columns:
            header = str(col[0].value or "")
            ws.column_dimensions[col[0].column_letter].width = min(max(len(header) + 4, 14), 55)
    if "priority_metric_review" in wb.sheetnames:
        ws = wb["priority_metric_review"]
        headers = {cell.value: cell.column_letter for cell in ws[1]}
        if "ra_decision" in headers:
            dv = DataValidation(type="list", formula1='"not_started,verified,use_with_caution,rejected,needs_page_check,needs_definition_check"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f"{headers['ra_decision']}2:{headers['ra_decision']}{ws.max_row}")
    wb.save(xlsx_path)
except Exception as e:
    print("Formatting skipped:", e)

print("Wrote", xlsx_path)
print("Wrote", priority_csv)
print("Wrote", summary_csv)

In [ ]:
summary_lines = []
summary_lines.append("# Comparator Metric Extraction Summary")
summary_lines.append("")
summary_lines.append(f"Created: {datetime.now().isoformat(timespec='seconds')}")
summary_lines.append("")
summary_lines.append(f"- Usable comparator full texts: {len(usable)}")
summary_lines.append(f"- Candidate metric rows: {len(candidates)}")
summary_lines.append(f"- Priority review rows: {len(priority_review)}")
summary_lines.append("")
summary_lines.append("## Metric Counts")
if candidates.empty:
    summary_lines.append("No candidate metrics found.")
else:
    for metric, count in candidates["metric"].value_counts().items():
        summary_lines.append(f"- {metric}: {count}")
summary_lines.append("")
summary_lines.append("## Source Counts")
if not candidates.empty:
    for sid, count in candidates["source_id"].value_counts().items():
        title = candidates.loc[candidates["source_id"].eq(sid), "title"].iloc[0]
        summary_lines.append(f"- {sid}: {count} candidates — {title}")
summary_lines.append("")
summary_lines.append("## Use Rule")
summary_lines.append("These are candidate extractions only. A number enters the paper only after the reviewer confirms the source quote, page/table, metric definition, direct-vs-total distinction, and projected-vs-realised status.")

summary_path = OUT_DIR / "COMPARATOR_EXTRACTION_SUMMARY.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print("Wrote", summary_path)
print("\n".join(summary_lines[:40]))